# Surgical Vision — run all experiments on Colab

1. Runtime → Change runtime type → **GPU** (T4 or better).
2. Course data must be in Google Drive under `MyDrive/BN5211/` (`Dataset/`, `dataset-full/`, `RARP_1FPS/`).
3. Results go to `MyDrive/BN5211/runs/`. Finished runs are skipped, so after a disconnect just run the setup cells again and re-run the project cell that was interrupted.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d repo && git -C repo pull --ff-only || git clone https://github.com/weirdquantum/surgical-vision-moco.git repo
%cd /content/repo
!pip install -q pyyaml

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/BN5211'
# Data on the local disk (fast I/O); runs on Drive (persistent).
!mkdir -p /content/data {DRIVE}/runs
!cp -rn {DRIVE}/Dataset {DRIVE}/dataset-full {DRIVE}/RARP_1FPS /content/data/
!ln -sfn /content/data data
!ln -sfn {DRIVE}/runs runs
!ls data
!python -m action_recognition.prepare --data-root data --workers 4

In [ ]:
!python -m pytest -q tests

## Project 1 — instrument classification

In [ ]:
!bash scripts/run_all.sh core p1 2>&1 | tee -a runs/colab_p1.log

## Project 2 — action recognition (longest)

In [ ]:
!bash scripts/run_all.sh core p2 2>&1 | tee -a runs/colab_p2.log

## Project 3 — MoCo pretraining + fine-tuning

In [ ]:
!bash scripts/run_all.sh core p3 2>&1 | tee -a runs/colab_p3.log

## Summaries

In [ ]:
!for f in runs/*/*/summary.md; do echo "== $f"; cat "$f"; done
!cat runs/contrastive_learning/probe/probe.json 2>/dev/null | head -50